# Zoom In: A Hands-On Introduction to Neural Network Circuits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jonny-English/learn-interpretability/blob/main/notebooks/circuits_zoom_in_en.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python 3.8+](https://img.shields.io/badge/python-3.8+-blue.svg)](https://www.python.org/downloads/)

> **[Chinese version (中文版)](circuits_zoom_in_zh.ipynb)** | This notebook reproduces the core ideas from Olah et al. 2020, *Zoom In: An Introduction to Circuits*.

**Table of Contents**
- [Section 0: Imports](#section-0-imports)
- [Section 1: Load Model](#section-1-load-model)
- [Section 2: Feature Visualization](#section-2-feature-visualization-activation-maximization)
- [Section 3: Dataset Validation](#section-3-validate-with-real-images)
- [Section 4: Orientation Tuning](#section-4-orientation-tuning-experiment)
- [Section 5: Circuit Analysis](#section-5-circuit-analysis)
- [Section 6: Universality](#section-6-universality-verification)
- [Section 7: Limitations](#section-7-limitations-and-open-questions)

---

The paper makes three core claims:

| Claim | Meaning |
|-------|--------|
| **Features** | Neurons detect human-understandable visual features (curves, textures, colors...) |
| **Circuits** | Features connect via weights to form circuits that perform specific computations |
| **Universality** | The same features and circuits appear across different models and tasks |

Model used: **InceptionV1 (Inception5h)**, consistent with the original paper.

---
### Model Layer Overview

```
Input image (224x224 pixels)
  conv2d0 -> conv2d1 -> conv2d2      <- Early conv layers, detect color/edge
  mixed3a -> mixed3b                <- Mid-shallow, curves and mid-level features
  mixed4a -> mixed4b -> ... -> mixed4e  <- Mid layers, textures/object parts
  mixed5a -> mixed5b                <- Deep layers, complex semantic features
  softmax outputs probabilities for 1000 classes
```

Each `mixed` block is an **Inception module** with four parallel branches (1x1, 3x3, 5x5 conv + pooling),
whose outputs are **concatenated** along the channel dimension.

## Section 0: Imports

In [ ]:
# Google Colab users: uncomment the line below to install required packages:
# !pip install torch torchvision torch-lucent matplotlib numpy Pillow

# torch: the core deep learning framework (tensors, autograd, neural network modules)
import torch
# torchvision: computer vision utilities — pretrained models, datasets, image transforms
import torchvision
# T: shorthand for the transforms module, used to preprocess images before feeding to the model
import torchvision.transforms as T
# numpy: numerical computing library, used for array operations and data manipulation
import numpy as np
# matplotlib: the main plotting library
import matplotlib
# pyplot: high-level interface for creating figures and plots
import matplotlib.pyplot as plt
# fm: font manager, used here to load a Chinese font for plot labels
import matplotlib.font_manager as fm
# gridspec: lets us create complex subplot layouts with custom row/column sizes
import matplotlib.gridspec as gridspec
# PIL Image: used to create synthetic arc stimuli (Section 4)
# PIL ImageDraw: used to draw shapes (arcs) onto those images
from PIL import Image, ImageDraw
import math, warnings, os
# Suppress deprecation and user warnings so the output stays clean
warnings.filterwarnings('ignore')

# ── CI mode: use minimal steps in GitHub Actions to verify code runs ──
# When running in CI (Continuous Integration), we use far fewer optimization steps
# so the test completes quickly (8 steps vs 512). Set CI=true in your environment to enable.
CI_MODE = os.environ.get('CI') == 'true'
默认步数 = 8 if CI_MODE else 512  # default_steps: number of gradient ascent iterations for feature viz

# ── Chinese font config (for plot labels) ──
# This function tries a list of common Chinese font paths across macOS, Linux, and Windows.
# If found, it registers the font with matplotlib so Chinese characters render correctly.
def _setup_chinese_font():
    for path in ['/System/Library/Fonts/STHeiti Medium.ttc',
                 '/System/Library/Fonts/PingFang.ttc',
                 '/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc',
                 '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
                 '/usr/share/fonts/wqy-zenhei/wqy-zenhei.ttc',
                 'C:/Windows/Fonts/msyh.ttc']:
        if os.path.exists(path):
            # Register the font file with matplotlib's font manager
            fm.fontManager.addfont(path)
            # Get the font's family name so we can set it as the default
            name = fm.FontProperties(fname=path).get_name()
            # Set this font as the first choice; fall back to DejaVu Sans if unavailable
            matplotlib.rcParams['font.sans-serif'] = [name, 'DejaVu Sans']
            return name
    return None  # No Chinese font found; Chinese characters will appear as boxes

_cn_font_name = _setup_chinese_font()
# Prevent matplotlib from rendering minus signs as boxes when using a Chinese font
matplotlib.rcParams['axes.unicode_minus'] = False

# lucent: a PyTorch port of the Tensorflow Lucid library for neural network visualization
# inceptionv1: loads the InceptionV1 (Inception5h) model pretrained on ImageNet
from lucent.modelzoo import inceptionv1
# render: runs the optimization loop (gradient ascent on the input image)
# objectives: defines what we're maximizing — e.g. the mean activation of a single channel
from lucent.optvis import render, objectives

# Select GPU if available; otherwise fall back to CPU.
# .to(device) later moves tensors and models onto whichever device is selected.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | PyTorch: {torch.__version__}')
print(f'Chinese font: {_cn_font_name or "not found"}')
if CI_MODE:
    print('⚡ CI mode: using minimal optimization steps, only verifying code runs')


def show_image_grid(images, titles, suptitle=None, cols=4, cell_size=3.2):
    """Display a list of images in a grid with titles."""
    # Calculate how many rows we need given the number of columns
    rows = math.ceil(len(images) / cols)
    # Create the grid of subplots; figsize scales with cols and rows
    fig, axes = plt.subplots(rows, cols, figsize=(cols * cell_size, rows * cell_size))
    # Flatten the 2D axes array so we can iterate over it with a single index
    axes = np.array(axes).flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        axes[i].imshow(img)   # Display the image in this subplot
        axes[i].axis('off')   # Hide axis ticks and border for a cleaner look
        axes[i].set_title(title, fontsize=8.5)
    # Turn off any unused subplot slots (when len(images) < rows*cols)
    for j in range(len(images), len(axes)):
        axes[j].axis('off')
    if suptitle:
        # Add an overall title above all subplots; y=1.01 prevents overlap with the top row
        plt.suptitle(suptitle, fontsize=12, y=1.01)
    plt.tight_layout()  # Automatically adjust spacing so titles/labels don't overlap
    plt.show()

## Section 1: Load Model

In [ ]:
# Load pretrained InceptionV1 model
# pretrained=True uses weights trained on ImageNet
# First run downloads ~27MB weights, cached locally afterwards
model = inceptionv1(pretrained=True).to(device).eval()
# .eval() switches to inference mode, disabling Dropout etc.
print('Model loaded')

# Test with a random noise image
with torch.no_grad():
    test_input = torch.randn(1, 3, 224, 224, device=device)  # 1 image, 3 channels (RGB), 224x224
    test_output = model(test_input)
print(f'Output shape: {test_output.shape}  <- 1 image, 1000 class scores')

In [ ]:
# Use hooks to inspect output shapes of each layer.
# A "forward hook" is a callback function that PyTorch automatically calls
# after a layer finishes its forward computation. We use this to record
# each layer's output shape without changing any model behaviour.

# The layers we want to inspect — spanning early, mid, and deep parts of InceptionV1
目标层 = ['conv2d0', 'conv2d1', 'conv2d2',
           'mixed3a', 'mixed3b', 'mixed4a', 'mixed4e', 'mixed5b']  # target_layers

形状记录 = {}  # shape_record: dict mapping layer name -> output tensor shape
钩子列表 = []  # hook_list: we keep references so we can remove hooks after use

# Iterate over all named modules (layers) in the model
for 层名, 模块 in model.named_modules():  # layer_name, module
    if 层名 in 目标层:
        # Define a hook function for this specific layer.
        # The default argument n=层名 captures the current value of 层名,
        # because Python closures capture variables by reference, not by value.
        def _记录形状(m, 输入, 输出, n=层名):  # _record_shape(m, input, output)
            # output.shape is (batch_size, channels, height, width) for conv layers
            形状记录[n] = tuple(输出.shape)
        # Register the hook on this module; it will fire on every forward pass
        钩子列表.append(模块.register_forward_hook(_记录形状))

# Run one forward pass with a dummy image to trigger all hooks.
# torch.no_grad() disables gradient computation — we only need the output values,
# not gradients, so this saves memory and speeds things up.
with torch.no_grad():
    model(torch.randn(1, 3, 224, 224, device=device))

# Remove all hooks now that we have the shapes we need.
# Always remove hooks when done — if left attached, they run on every future forward
# pass and can slow things down or cause unexpected side-effects.
for 钩子 in 钩子列表:  # hook
    钩子.remove()

print('Layer -> Output shape (batch, channels, height, width)')
print('(More channels = more features; deeper layers have smaller H/W but more channels)')
print()
for 名称 in 目标层:  # name
    形状 = 形状记录[名称]  # shape
    print(f'  {名称:12s}: {形状}')

---
## Section 2: Feature Visualization (Activation Maximization)

**Question**: What is a neuron actually "looking" for?

**Method**: Start from a random noise image, use gradient ascent to iteratively modify it,
maximizing the target neuron's activation. The resulting image is the neuron's "preferred stimulus".

```
Start: random noise image
  | Iterate: modify image in the direction that makes the neuron "more excited"
Result: the pattern the neuron most wants to see
```

lucent adds two tricks for cleaner results:
- **Fourier parameterization**: optimize in frequency domain to avoid high-frequency noise
- **Random transform augmentation**: randomly scale/rotate each step for robustness

In [ ]:
def 可视化神经元(层名, 通道, 步数=默认步数, 显示=True, 标题=None):  # visualize_neuron(layer, channel, steps, show, title)
    """
    Feature visualization (activation maximization) for a given layer and channel.
    More steps = clearer result but slower. ~30 sec for 512 steps on CPU.
    Returns numpy array (128, 128, 3), pixel values [0,1]
    """
    # Define the optimization objective: maximize the mean activation of the specified channel
    # in the specified layer. This tells lucent "make the image excite this neuron as much as possible."
    目标 = objectives.channel(层名, 通道)  # target

    # render_vis performs gradient ascent on the input image:
    #   - Starts from a random noise image
    #   - Computes the gradient of the objective with respect to each pixel
    #   - Updates the pixels in the direction that increases the objective (gradient ascent)
    #   - Repeats for `thresholds` steps
    # thresholds=(步数,) is a tuple specifying at which step(s) to save a snapshot
    # show_inline=False and progress=False suppress intermediate output
    结果列表 = render.render_vis(  # result_list
        model, 目标,
        thresholds=(步数,),
        show_inline=False,
        progress=False,
    )

    # result_list has one entry per threshold; each entry is a batch of images.
    # [0][0] takes: first (and only) threshold snapshot -> first image in the batch
    # This gives us a numpy array of shape (128, 128, 3) with values in [0, 1]
    图片 = 结果列表[0][0]  # image

    if 显示:
        plt.figure(figsize=(3.5, 3.5))
        plt.imshow(图片)           # Show the optimized image
        plt.axis('off')            # Hide axes for a clean display
        plt.title(标题 or f'{层名}:{通道}', fontsize=10)
        plt.tight_layout()
        plt.show()
    return 图片  # Return the image so callers can cache or further process it

print('Generating feature visualization for mixed3b:379 (~30 sec)...')
_ = 可视化神经元('mixed3b', 379, 步数=默认步数, 标题='mixed3b:379 Curve Detector')

In [ ]:
# Batch-generate visualizations for 8 representative neurons
#
# mixed3b has 480 channels, four branches concatenated:
#   channels   0 ~ 127  <- 1x1 branch
#   channels 128 ~ 319  <- 3x3 branch
#   channels 320 ~ 415  <- 5x5 branch
#   channels 416 ~ 479  <- pooling branch

# Each tuple is (layer_name, channel_index, human_readable_description)
# We hand-pick neurons at different depths and branches to illustrate the hierarchy.
代表性神经元 = [  # representative_neurons
    # (layer,       channel, description)
    ('conv2d0',    0,    'Early: color/edge'),
    ('conv2d1',    5,    'Early: color contrast'),
    ('mixed3a',    0,    'Mid: edge detector'),
    ('mixed3b',    6,    'High/low freq'),
    ('mixed3b',  379,    'Curve detector'),
    ('mixed3b',  425,    'Pooling branch'),
    ('mixed4a',   22,    'Mid: texture'),
    ('mixed4e',  254,    'Deep: complex'),
]

print(f'{len(代表性神经元)} neurons, ~5-10 min on CPU')
print()

# 缓存 (cache): a dict mapping (layer, channel) -> (image_array, description).
# We store results here so we can reuse the expensive visualizations in later cells
# without rerunning the optimization.
缓存 = {}  # cache

for 层名, 通道号, 描述 in 代表性神经元:  # layer, channel, description
    print(f'  Processing {层名}:{通道号} ({描述})', end=' ... ', flush=True)
    # Run activation maximization with 显示=False so we don't pop up 8 separate windows
    图片 = 可视化神经元(层名, 通道号, 步数=默认步数, 显示=False)  # image
    # Store both the image and its description under the (layer, channel) key
    缓存[(层名, 通道号)] = (图片, 描述)
    print('done')

print('\nAll done!')

In [ ]:
# Pull the image arrays and labels from the cache in the same order as 代表性神经元,
# then display them all in a single grid for easy comparison.
show_image_grid(
    [缓存[(层, 通道)][0] for 层, 通道, _ in 代表性神经元],   # list of image arrays
    [f'{层}:{通道}\n{描述}' for 层, 通道, 描述 in 代表性神经元],  # matching title for each image
    suptitle='Feature Visualization (Activation Maximization)\nShallow -> Deep: Simple -> Complex',
)

print()
print('Key observations:')
print('  conv2d0/1  -> stripes, color gradients — the most basic visual elements')
print('  mixed3a/3b -> directional curves and textures emerge')
print('  mixed4a/4e -> complex repeating textures, even material-like or object parts')

---
## Section 3: Validate with Real Images

The previous section showed "synthetic" images — optimal stimuli optimized from noise.
Now we take a different angle: in a real image dataset, **which images activate this neuron most strongly?**

If the synthetic and real images look similar visually -> the neuron is detecting a genuine
visual pattern in the real world, not an artifact of the optimization process.

In [ ]:
from torch.utils.data import DataLoader

# Use CIFAR-10 test set as real image source
# CIFAR-10: 10 classes (airplane/car/bird/cat etc.), original size 32x32, ~30MB
# Downside: low resolution, blurry when resized to 224x224, but enough to demonstrate the concept

# T.Compose chains multiple transforms that will be applied in sequence to every image.
图片变换 = T.Compose([  # image_transform
    # Upscale from CIFAR-10's native 32x32 to 224x224 — the size InceptionV1 expects
    T.Resize(224),
    # Convert the PIL image to a float32 tensor with shape (3, H, W), values in [0, 1]
    T.ToTensor(),
    # Normalize each channel using ImageNet statistics (mean and std).
    # Why? The pretrained InceptionV1 was trained on data normalized this way,
    # so we must apply the same normalization at inference time.
    # Formula: normalized = (pixel - mean) / std
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# Download and load the CIFAR-10 test split.
# The path stores the downloaded data; train=False selects the 10,000-image test set.
数据集 = torchvision.datasets.CIFAR10(  # dataset
    os.path.join(os.path.dirname(os.getcwd()), 'data', 'cifar10'),
    train=False,
    download=True,
    transform=图片变换   # apply our transform pipeline to every image on load
)

# DataLoader wraps the dataset and handles batching and shuffling.
# batch_size=64 means we process 64 images at once per forward pass (faster than one-by-one).
# shuffle=False preserves ordering so results are reproducible.
# num_workers=0 loads data on the main thread (avoids multiprocessing issues on some systems).
数据加载器 = DataLoader(数据集, batch_size=64, shuffle=False, num_workers=0)  # data_loader
print(f'CIFAR-10 test set loaded, {len(数据集)} images total')

In [ ]:
def 反归一化(张量):  # denormalize(tensor)
    """Reverse normalization to get displayable image.
    Input: tensor shape (3, H, W)
    Output: numpy array shape (H, W, 3), pixel range [0, 1]
    """
    # Recreate the per-channel mean and std used during normalization.
    # .view(3, 1, 1) reshapes them to (channels, 1, 1) so they broadcast
    # correctly when multiplied against a (3, H, W) image tensor.
    均值 = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)  # mean
    标准差 = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)  # std
    # Reverse the normalization: pixel = normalized * std + mean
    # .cpu() moves the tensor off the GPU (if any) so we can convert to numpy.
    # .clamp(0, 1) clips values to [0, 1] to avoid display glitches from floating-point rounding.
    还原后 = (张量.cpu() * 标准差 + 均值).clamp(0, 1)  # restored
    # .permute(1, 2, 0) rearranges from (C, H, W) to (H, W, C) — the format matplotlib expects
    return 还原后.permute(1, 2, 0).numpy()


def 找最高激活图片(模型, 数据加载器, 层名, 通道, 取前K=9, 最多批次=10):  # find_top_activating_images(model, loader, layer, channel, top_k, max_batches)
    """Find images that most strongly activate a given neuron."""
    激活值列表 = []  # activation_list: stores per-image activation scalars, batch by batch
    图片列表 = []   # image_list: stores the corresponding image tensors
    暂存 = {}       # store: temporary dict used to pass data out of the hook closure

    # Hook function: called automatically after the target layer's forward pass.
    # m = the module, 输入 = its input tensor(s), 输出 = its output tensor.
    # output shape is (batch, channels, H, W); we take channel `通道` and average spatially.
    def 记录激活(m, 输入, 输出):  # record_activation
        # For each image in the batch, compute the mean activation of the target channel
        # over its spatial dimensions (H, W). This gives one scalar per image.
        暂存['激活'] = 输出[:, 通道].mean((1, 2)).detach().cpu()

    # Register the hook on the target layer; dict(named_modules()) is a quick lookup by name
    钩子 = dict(模型.named_modules())[层名].register_forward_hook(记录激活)  # hook

    with torch.no_grad():
        for 批次序号, (图片批, _) in enumerate(数据加载器):  # batch_idx, (image_batch, _)
            if 批次序号 >= 最多批次:
                break  # Stop after 最多批次 batches to keep it fast (not all 10,000 images)
            # Move image batch to the correct device (CPU or GPU)
            模型(图片批.to(device))      # forward pass — triggers the hook, populates 暂存
            激活值列表.append(暂存['激活'])  # save this batch's activation scalars
            图片列表.append(图片批)          # save the original (CPU) image tensors

    # Always remove the hook when done to avoid side-effects on future forward passes
    钩子.remove()

    # Concatenate all batches into single tensors across the batch dimension
    所有激活 = torch.cat(激活值列表)  # all_activations, shape (N,)
    所有图片 = torch.cat(图片列表)    # all_images, shape (N, 3, 224, 224)

    # .topk(K) returns the K largest values and their indices.
    # We use the indices to select the corresponding images.
    最高索引 = 所有激活.topk(取前K).indices  # top_indices
    return 所有图片[最高索引], 所有激活[最高索引]


# Search for mixed3b:379 (curve detector)
目标层名 = 'mixed3b'  # target_layer
目标通道 = 379        # target_channel (the curve detector we identified earlier)
print(f'Searching CIFAR-10 for images that most activate {目标层名}:{目标通道}...')
最高图片, 最高激活值 = 找最高激活图片(model, 数据加载器, 目标层名, 目标通道)  # top_images, top_activations
print('Search complete')

In [ ]:
# Retrieve the cached synthetic (activation-maximized) image for the target neuron
合成图, _ = 缓存[(目标层名, 目标通道)]  # synthetic_image

# Create a figure with a custom GridSpec layout:
#   - Columns 0-1: synthetic image (spans all 3 rows, wider)
#   - Column 2:    "vs" separator text
#   - Columns 3-5: 3x3 grid of top real images (one per cell)
图 = plt.figure(figsize=(14, 5.5))  # fig
网格 = gridspec.GridSpec(3, 7, figure=图, wspace=0.08, hspace=0.08)  # grid

# Left panel: the synthetic activation-maximization image, spanning all 3 rows and 2 columns
左图 = 图.add_subplot(网格[:, :2])  # left_plot
左图.imshow(合成图)
左图.axis('off')
左图.set_title(f'Feature Viz\n(Activation Max)\n{目标层名}:{目标通道}', fontsize=10)

# Middle column: a visual separator showing "vs"
中图 = 图.add_subplot(网格[:, 2])  # mid_plot
中图.text(0.5, 0.5, 'vs', ha='center', va='center', fontsize=20, color='gray', transform=中图.transAxes)
中图.axis('off')

# Right panel: 9 real CIFAR-10 images arranged in a 3×3 grid (columns 3-6)
for i in range(9):
    # divmod(i, 3) gives (row_index, col_index) for a 3×3 layout
    行, 列 = divmod(i, 3)  # row, col
    子图 = 图.add_subplot(网格[行, 列+3])  # subplot; +3 offsets into the right portion of the grid
    子图.imshow(反归一化(最高图片[i]))    # undo normalization before displaying
    子图.axis('off')
    子图.set_title(f'Act:{最高激活值[i]:.2f}', fontsize=7)  # show activation strength as subtitle

# Overall title for the right section, positioned at the top center of that area
图.text(0.67, 1.0, 'Top 9 real images by activation (CIFAR-10)', ha='center', fontsize=11)
plt.show()

print()
print('How to interpret:')
print('  Left = the neuron\'s "ideal" pattern (synthetic)')
print('  Right = real images that excite it most')
print('  Visual similarity -> the neuron detects a genuine real-world feature')
print('  Note: this is a qualitative comparison. Quantitative methods (e.g. cosine similarity) could further validate; left for future work.')

---
## Section 4: Orientation Tuning Experiment

The original paper shows: curve detectors respond most strongly to curves at **specific orientations**,
with activation dropping as the curve is rotated away.
This is highly consistent with **direction selectivity** in simple cells of the primate visual cortex (V1).

**Method**: Generate arcs at different orientations, measure activation for each, plot as polar diagram.
The prominent "peak" direction on the polar plot = the neuron's preferred curve orientation.

In [ ]:
def 生成弧线图片(角度, 图片尺寸=224, 半径=60, 线宽=5):  # generate_arc_image(angle, image_size, radius, width)
    """Draw a white arc on gray background, return normalized tensor (1,3,224,224)"""
    # Create a square RGB image filled with mid-gray (128,128,128).
    # Mid-gray is chosen as a neutral background that doesn't bias activations.
    图片 = Image.new('RGB', (图片尺寸, 图片尺寸), (128, 128, 128))  # image
    画笔 = ImageDraw.Draw(图片)  # draw: a drawing context that lets us paint shapes onto 图片
    cx, cy = 图片尺寸 // 2, 图片尺寸 // 2   # center coordinates of the image

    # PIL's arc() draws a portion of an ellipse inscribed in a bounding box.
    # We draw an arc centered at (cx, cy) spanning ±60 degrees around the given angle,
    # producing a 120-degree white arc at the desired orientation.
    外框 = [cx-半径, cy-半径, cx+半径, cy+半径]  # bbox: [left, top, right, bottom]
    画笔.arc(外框, start=角度-60, end=角度+60, fill=(255, 255, 255), width=线宽)

    # Apply the same preprocessing pipeline used during training so the model
    # receives input in the expected format (normalized float32 tensor).
    变换 = T.Compose([  # transform
        T.ToTensor(),                                          # PIL -> (3, H, W) float32 in [0,1]
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet normalization
    ])
    # .unsqueeze(0) adds a batch dimension: (3, H, W) -> (1, 3, H, W)
    # The model always expects a batch dimension even for a single image.
    return 变换(图片).unsqueeze(0)

# Preview 6 evenly-spaced arc orientations so we can visually verify the stimuli look correct
图, 子图数组 = plt.subplots(1, 6, figsize=(12, 2.2))  # fig, axes: 1 row, 6 columns
for 子图, 角度 in zip(子图数组, range(0, 360, 60)):  # ax, angle: iterate over 0°, 60°, 120°, ...
    # 生成弧线图片 returns shape (1,3,H,W); [0] removes the batch dim -> (3,H,W) for 反归一化
    子图.imshow(反归一化(生成弧线图片(角度)[0]))
    子图.axis('off')
    子图.set_title(f'{角度} deg', fontsize=9)

plt.suptitle('Synthetic arc stimuli (6 orientations, 60 deg apart)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
def 测量方向调谐(模型, 层名, 通道, 角度数=36):  # measure_orientation_tuning(model, layer, channel, n_angles)
    """Rotate arc through 360 degrees, measure activation at each angle.
    Returns: angle_array, activation_array (both numpy)
    """
    # Create 36 evenly-spaced angles from 0 to 360 degrees (endpoint=False avoids duplicating 0°/360°)
    角度数组 = np.linspace(0, 360, 角度数, endpoint=False)  # angles
    激活值列表 = []  # activation_list: one scalar per angle
    暂存 = {}        # store: passes data from inside the hook closure back to the outer function

    # Hook that captures the scalar activation of the target channel for the current image
    def 记录(m, 输入, 输出):  # record
        # output shape: (1, channels, H, W) — single image, so batch dim = 1
        # .mean() averages over the spatial (H, W) dimensions -> one scalar
        # .item() converts the 0-d tensor to a plain Python float
        暂存['值'] = 输出[0, 通道].mean().item()  # value

    # Register the hook on the named layer
    钩子 = dict(模型.named_modules())[层名].register_forward_hook(记录)  # hook

    with torch.no_grad():
        for 角度 in 角度数组:  # angle: step through every test orientation
            # Generate the arc at this angle, run the forward pass (triggers the hook)
            模型(生成弧线图片(角度).to(device))
            激活值列表.append(暂存['值'])  # save the activation recorded by the hook

    # Remove the hook now that all angles have been measured
    钩子.remove()
    return 角度数组, np.array(激活值列表)  # return angles and activations as numpy arrays


# Neurons to test for orientation tuning
待测神经元 = [('mixed3b', 379), ('mixed3b', 425)]  # neurons_to_test
print(f'Measuring orientation tuning ({36*len(待测神经元)} forward passes)...')

调谐结果 = []  # tuning_results: list of (angles, activations, label) for plotting
for 层名, 通道 in 待测神经元:
    角度数组, 激活值数组 = 测量方向调谐(model, 层名, 通道)  # angles, activations
    调谐结果.append((角度数组, 激活值数组, f'{层名}:{通道}'))
    # argmax() finds the index of the highest activation, then we look up the corresponding angle
    偏好方向 = 角度数组[激活值数组.argmax()]  # preferred_direction
    print(f'  {层名}:{通道} preferred direction = {偏好方向:.0f} deg')

In [ ]:
# Create one polar subplot per neuron.
# projection='polar' makes matplotlib use a circular coordinate system
# where the angle = stimulus orientation and the radius = activation strength.
图, 子图数组 = plt.subplots(
    1, len(调谐结果),
    figsize=(5*len(调谐结果), 4.5),
    subplot_kw={'projection': 'polar'}
)
# If there is only one neuron, plt.subplots returns a single Axes object, not a list.
# Wrapping it in a list keeps the rest of the code consistent.
if len(调谐结果) == 1:
    子图数组 = [子图数组]

for 子图, (角度数组, 激活值数组, 标签) in zip(子图数组, 调谐结果):
    # Normalize activations to [0, 1] so all neurons use the same radial scale.
    # Adding 1e-8 prevents division by zero when max == min (flat tuning curve).
    归一化激活 = (激活值数组 - 激活值数组.min()) / (激活值数组.max() - 激活值数组.min() + 1e-8)  # normalized

    # Polar plots require angles in radians; np.deg2rad converts degrees -> radians.
    # np.append(..., [0]) closes the loop by repeating the first point at the end,
    # so the line connects back to the start instead of leaving a gap.
    弧度数组 = np.append(np.deg2rad(角度数组), np.deg2rad(角度数组[0]))  # radians
    值数组 = np.append(归一化激活, 归一化激活[0])  # values (closed loop)

    # Draw the tuning curve line
    子图.plot(弧度数组, 值数组, linewidth=2, color='steelblue')
    # Fill the area under the curve to make the preferred direction visually obvious
    子图.fill(弧度数组, 值数组, alpha=0.25, color='steelblue')
    子图.set_title(标签, pad=15)
    # set_theta_zero_location('N') puts 0 degrees at the top (North), matching compass convention
    子图.set_theta_zero_location('N')
    # set_theta_direction(-1) makes angles increase clockwise (matches image/compass convention)
    子图.set_theta_direction(-1)

plt.suptitle('Orientation Tuning (Polar Plot)\nPeak = preferred direction, radius = normalized activation', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('Conclusion: curve detectors respond significantly more to arcs at specific orientations,')
print('consistent with V1 simple cell behavior in visual neuroscience.')

---
## Section 5: Circuit Analysis — How Are Curves "Computed"?

**The circuits claim**: The curve detector doesn't detect curves out of thin air.
It **reads the outputs of upstream edge detectors** and combines them through learned weights.

```
mixed3a edge detectors  ──  weight matrix  ──>  mixed3b curve detector
(detect edges at various orientations)          (detect curves at specific orientation)
```

We verify this circuit by reading the weight matrices directly:
find the most influential upstream neurons, then visualize them.

### Which branch does mixed3b:379 belong to?
- Channel 379 falls in the 5x5 branch (range 320~415)
- Branch-internal index = 379 - 320 = **59**
- 5x5 branch structure: `mixed3a(256ch)` -> `1x1 bottleneck(32ch)` -> `5x5 conv(96ch)`

In [ ]:
# ── Read weights, analyze upstream dependencies of mixed3b:379 ──

目标曲线通道 = 379  # target_curve_channel: the curve detector neuron we've been studying
# Channel 379 belongs to the 5x5 branch (channels 320–415).
# Its index within that branch is 379 - 320 = 59.
分支内索引 = 目标曲线通道 - 320  # branch_internal_index = 59

# Retrieve the learned weight tensors directly from the model's parameters.
# .data gives the raw tensor without autograd tracking; .cpu() moves it off GPU for numpy ops.

# 1x1 bottleneck: compresses mixed3a's 256 channels down to 32, reducing computation
#    weight shape: (out_channels=32, in_channels=256, kernel_h=1, kernel_w=1)
权重_压缩 = model.mixed3b_5x5_bottleneck_pre_relu_conv.weight.data.cpu()  # weights_bottleneck

# 5x5 convolution: operates on the 32 bottleneck channels to produce 96 output channels
#    weight shape: (out_channels=96, in_channels=32, kernel_h=5, kernel_w=5)
权重_5x5 = model.mixed3b_5x5_pre_relu_conv.weight.data.cpu()  # weights_5x5

print(f'1x1 bottleneck weight shape: {tuple(权重_压缩.shape)}')
print(f'  Meaning: 32 output channels, each connected to all 256 mixed3a channels')
print()
print(f'5x5 conv weight shape: {tuple(权重_5x5.shape)}')
print(f'  Meaning: 96 output channels, each with a 5x5 filter over 32 bottleneck channels')

# ── Step 1: Find which bottleneck channels matter most to our target neuron ──
# 目标滤波器 is the 5x5 filter for channel index `分支内索引` (our curve detector).
# Shape: (32, 5, 5) — one 5×5 kernel per bottleneck channel.
目标滤波器 = 权重_5x5[分支内索引]  # target_filter

# For each of the 32 bottleneck channels, sum the absolute weight values across
# the 5×5 spatial kernel. Larger sum -> that bottleneck channel influences our neuron more.
# .abs() ensures positive and negative weights both count as "strong" connections.
压缩通道重要性 = 目标滤波器.abs().sum((1, 2))  # bottleneck_importance, shape (32,)

# ── Step 2: Trace back from bottleneck channels to mixed3a's 256 channels ──
# .squeeze() removes the kernel dimensions (h=1, w=1), giving shape (32, 256)
# Row i of this matrix = how much each of the 256 mixed3a channels contributes to bottleneck channel i
压缩层权重矩阵 = 权重_压缩.squeeze()  # bottleneck_weight_matrix, shape (32, 256)

# Combine the two steps:
#   - 压缩通道重要性[:, None]  shape (32, 1):  how much each bottleneck channel matters to the curve detector
#   - 压缩层权重矩阵.abs()     shape (32, 256): how much each mixed3a channel contributes to each bottleneck channel
#   - Element-wise multiply then sum over the 32 bottleneck channels -> shape (256,)
# Result: one importance score per mixed3a channel, measuring its total indirect influence on mixed3b:379
mixed3a通道重要性 = (压缩通道重要性[:, None] * 压缩层权重矩阵.abs()).sum(0)  # mixed3a_channel_importance, shape (256,)

K = 8  # number of top upstream channels to examine
前K重要通道 = mixed3a通道重要性.topk(K)  # top_k_channels: .indices and .values attributes
上游通道索引 = 前K重要通道.indices.tolist()  # upstream_indices: list of channel numbers
上游重要性值 = 前K重要通道.values.tolist()   # upstream_importance_values

print()
print(f'Top {K} most influential upstream neurons for mixed3b:{目标曲线通道} (curve detector):')
for 排名, (索引, 重要性) in enumerate(zip(上游通道索引, 上游重要性值)):  # rank, (index, importance)
    print(f'  #{排名+1}: mixed3a:{索引:3d}  importance = {重要性:.3f}')

# ── Visualize importance distribution of all 256 upstream channels ──
plt.figure(figsize=(12, 3))
# Sort channels from most to least important so the bar chart is easy to read
排序索引 = mixed3a通道重要性.argsort(descending=True)  # sorted_indices
排序值 = mixed3a通道重要性[排序索引].numpy()            # sorted_values
# Color the top K bars blue; all others gray to highlight the sparse connectivity
颜色 = ['steelblue' if i < K else 'lightgray' for i in range(len(排序值))]  # colors
plt.bar(range(len(排序值)), 排序值, color=颜色, width=1.0)
plt.xlabel('mixed3a channels (sorted by importance)')
plt.ylabel('Combined importance')
plt.title(f'Importance of all 256 mixed3a channels for mixed3b:{目标曲线通道}\nBlue = top {K}', fontsize=10)
plt.tight_layout()
plt.show()
print('Observation: a few channels have high importance, most contribute near zero -> sparse connectivity')

In [ ]:
取前几个 = 4  # n_top: how many upstream neurons to visualize (the 4 most important)

print(f'Generating feature viz for top {取前几个} upstream neurons (~2 min)...')
上游图片列表 = []  # upstream_images: list of (channel_index, image_array) tuples
for 通道 in 上游通道索引[:取前几个]:  # channel: iterate over the top-K upstream channels
    print(f'  mixed3a:{通道}', end=' ... ', flush=True)
    图片 = 可视化神经元('mixed3a', 通道, 步数=默认步数, 显示=False)  # image
    上游图片列表.append((通道, 图片))
    print('done')

# Retrieve the cached curve detector image to show as the "output" of the circuit
曲线探测器图片, _ = 缓存[('mixed3b', 目标曲线通道)]  # curve_detector_image

# Layout: [upstream #1] [upstream #2] ... [upstream #N] [arrow] [curve detector]
# The arrow column is narrower (0.4x) to avoid a large empty gap between upstream and target.
宽度比例 = [1]*取前几个 + [0.4] + [1]  # width_ratios: one unit per upstream image, 0.4 for arrow, 1 for target
图 = plt.figure(figsize=((取前几个+1.4)*3.2, 3.5))
# GridSpec gives us manual control over column widths via width_ratios
网格 = gridspec.GridSpec(1, 取前几个+2, width_ratios=宽度比例, wspace=0.15)

# Plot each upstream (edge detector) neuron's feature visualization
for i, (通道, 图片) in enumerate(上游图片列表):
    子图 = 图.add_subplot(网格[0, i])
    子图.imshow(图片)
    子图.axis('off')
    子图.set_title(f'Upstream #{i+1}\nmixed3a:{通道}\n(edge detector)', fontsize=8.5)

# Arrow column: a text label indicating the direction of information flow (weighted sum -> curve detector)
箭头图 = 图.add_subplot(网格[0, 取前几个])  # arrow_plot
箭头图.text(0.5, 0.5, 'Weighted\nSum\n\u279c',  # \u279c is the unicode arrow character ➜
    ha='center', va='center', fontsize=13, color='gray',
    transform=箭头图.transAxes)
箭头图.axis('off')

# Final column: the curve detector — the output of combining the upstream edge detectors
目标图 = 图.add_subplot(网格[0, 取前几个+1])  # target_plot
目标图.imshow(曲线探测器图片)
目标图.axis('off')
目标图.set_title(f'Target: Curve Detector\nmixed3b:{目标曲线通道}', fontsize=8.5)

plt.suptitle('Circuit: Edge Detectors (mixed3a) \u279c Curve Detector (mixed3b)', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('This is a complete "circuit": upstream edge detectors -> weighted combination -> curve detector')

---
## Section 6: Universality Verification

**The universality claim**: Edge detectors, curve detectors and similar features are not unique to InceptionV1;
they emerge spontaneously in other architectures too.

We verify with **ResNet-18**:
- Completely different architecture (residual network vs Inception modules)
- Independently trained (different random seeds, possibly different training procedures)
- But also trained on ImageNet

In [ ]:
# Load ResNet-18 with ImageNet-pretrained weights.
# weights='IMAGENET1K_V1' is the torchvision v2 way to specify the weight set.
# .to(device): move all model parameters to GPU if available, otherwise CPU.
# .eval(): switch to inference mode — disables Dropout layers and makes BatchNorm
#          use its running statistics rather than batch statistics.
resnet模型 = torchvision.models.resnet18(weights='IMAGENET1K_V1').to(device).eval()  # resnet_model
print('ResNet-18 loaded')

# Access the weights of ResNet-18's very first convolutional layer (conv1).
# Shape: (out_channels=64, in_channels=3, kernel_h=7, kernel_w=7)
# Each of the 64 filters is a 7×7 RGB kernel — we can visualize these directly
# because they operate on raw pixel values (3 RGB channels).
第一层权重 = resnet模型.conv1.weight.data.cpu()  # first_layer_weights, (64, 3, 7, 7)

# Plot all 64 filters in a 4×8 grid
图, 子图数组 = plt.subplots(4, 8, figsize=(12, 6))
for i, 子图 in enumerate(子图数组.flatten()):
    # Extract the i-th filter and rearrange from (3,7,7) to (7,7,3) for imshow
    滤波器 = 第一层权重[i].permute(1, 2, 0).numpy()  # filter: (H, W, C) layout

    # Normalize each filter independently to [0, 1] so small-valued filters are still visible.
    # Without this, many filters would appear nearly black because their raw values are tiny.
    # Adding 1e-8 prevents division by zero for a constant filter.
    滤波器 = (滤波器 - 滤波器.min()) / (滤波器.max() - 滤波器.min() + 1e-8)
    子图.imshow(滤波器)
    子图.axis('off')
    子图.set_title(str(i), fontsize=6)  # label each filter with its channel index

plt.suptitle('ResNet-18 first-layer filters (64 x 7x7, direct weight visualization)', fontsize=10)
plt.tight_layout()
plt.show()
print('Note: edge/color/orientation filters visible, similar to InceptionV1 early layers')

In [ ]:
# Method 2: Activation maximization on ResNet-18 (same method as InceptionV1)
# We apply the exact same gradient ascent technique to ResNet-18's intermediate layers
# to see what visual patterns those neurons prefer.

# Each tuple: (layer_name_in_resnet, channel_index, human_readable_label)
# ResNet layers are named layer1, layer2, layer3, layer4 — each containing residual blocks.
resnet待可视化神经元 = [  # resnet_neurons_to_visualize
    ('layer1',  0,  'ResNet layer1:0'),
    ('layer1',  8,  'ResNet layer1:8'),
    ('layer1', 16,  'ResNet layer1:16'),
    ('layer2',  0,  'ResNet layer2:0'),
    ('layer3',  0,  'ResNet layer3:0'),
]

print(f'Feature visualization on ResNet-18 ({len(resnet待可视化神经元)} neurons)...')
resnet可视化结果 = []  # resnet_viz_results: list of (image_array, label) tuples
for 层名, 通道, 标签 in resnet待可视化神经元:  # layer, channel, label
    print(f'  {标签}', end=' ... ', flush=True)
    # Build the objective: maximize the mean activation of `通道` in `层名`
    目标 = objectives.channel(层名, 通道)
    # Run gradient ascent optimization — same API as for InceptionV1
    结果 = render.render_vis(
        resnet模型, 目标,
        thresholds=(默认步数,),   # save a snapshot at this step count
        show_inline=False,
        progress=False
    )
    # 结果[0][0]: first threshold snapshot, first image in the batch -> (128, 128, 3) numpy array
    resnet可视化结果.append((结果[0][0], 标签))
    print('done')
print('All done')

In [ ]:
# Select four InceptionV1 neurons for the top row of the comparison grid,
# pulling their visualizations from the cache populated in Section 2.
inception对比图 = [  # inception_comparison: list of (image_array, label) tuples
    (缓存[('conv2d0',   0)][0],  'InceptionV1\nconv2d0:0\n(early)'),
    (缓存[('mixed3a',   0)][0],  'InceptionV1\nmixed3a:0\n(edge detector)'),
    (缓存[('mixed3b', 379)][0],  'InceptionV1\nmixed3b:379\n(curve detector)'),
    (缓存[('mixed3b',   6)][0],  'InceptionV1\nmixed3b:6\n(high/low freq)'),
]

# Use the larger of the two lists as the column count so neither row has unused slots
列数 = max(len(inception对比图), len(resnet可视化结果))  # n_cols
# Row 0 = InceptionV1 neurons; Row 1 = ResNet-18 neurons
图, 子图数组 = plt.subplots(2, 列数, figsize=(列数*3, 6.5))

# Top row: InceptionV1 feature visualizations
for i, (图片, 标签) in enumerate(inception对比图):
    子图数组[0, i].imshow(图片)
    子图数组[0, i].axis('off')
    子图数组[0, i].set_title(标签, fontsize=8)
# Hide any unused columns in the top row
for j in range(len(inception对比图), 列数):
    子图数组[0, j].axis('off')

# Bottom row: ResNet-18 feature visualizations
for i, (图片, 标签) in enumerate(resnet可视化结果):
    子图数组[1, i].imshow(图片)
    子图数组[1, i].axis('off')
    子图数组[1, i].set_title(标签, fontsize=8)
# Hide any unused columns in the bottom row
for j in range(len(resnet可视化结果), 列数):
    子图数组[1, j].axis('off')

plt.suptitle('Universality: InceptionV1 (top) vs ResNet-18 (bottom)\nDifferent architectures, independent training, yet similar early features', fontsize=11)
plt.tight_layout()
plt.show()

print()
print('Conclusion: two completely different architectures, trained independently, both develop similar edge/curve features.')
print()
print('Note: exact neuron-to-neuron correspondence is difficult because:')
print('  1. Different models have different channel orderings')
print('  2. Polysemanticity: a single neuron may respond to multiple unrelated features')
print('  3. Universality manifests at the "feature type" level, not specific channel numbers')

---
## Section 7: Limitations and Open Questions

This tutorial demonstrates the core ideas of the Circuits paper, but there are important limitations worth discussing.

In [ ]:
print('=' * 60)
print('Limitations and Open Questions')
print('=' * 60)
print()
print('1. Dataset Resolution')
print('   This tutorial uses CIFAR-10 (32x32 upscaled to 224x224), which is low-res.')
print('   Using ImageNet subsets or higher-resolution datasets would give cleaner validation.')
print()
print('2. Polysemanticity')
print('   This tutorial assumes each neuron corresponds to a clean feature, but in reality')
print('   many neurons respond to multiple unrelated features (polysemanticity).')
print('   This is the core research question of Toy Models of Superposition (Elhage et al., 2022).')
print()
print('3. Simplified Circuit Analysis')
print('   Section 5 uses absolute weight magnitude as a proxy for "influence",')
print('   but this ignores ReLU nonlinearities, feature interactions, and input distribution.')
print('   More rigorous methods include activation patching and causal scrubbing.')
print()
print('4. From Vision to Language')
print('   This tutorial focuses on vision models (CNNs), but the frontier of')
print('   interpretability research has shifted to Transformer language models.')
print('   Key concepts include:')
print('   - Induction Heads: a classic Transformer circuit')
print('   - Sparse Autoencoders (SAE): a new approach to polysemanticity')
print('   - TransformerLens: a Python toolkit for analyzing GPT-style models')
print()
print('These limitations point precisely to the most active directions in interpretability research.')

---
## Summary

| Claim | Experiments in this notebook | Key conclusion |
|-------|---------------------------|----------------|
| **Features** | S2: Activation maximization<br>S3: Dataset validation | Neurons detect understandable visual patterns, consistent with real images |
| **Circuits** | S4: Orientation tuning<br>S5: Weight analysis | Curve detector = weighted combination of upstream edge detectors |
| **Universality** | S6: Cross-architecture comparison | Different models independently develop similar basic features |
| **Limitations** | S7: Open questions | Polysemanticity, nonlinear interactions, extension to Transformers |

---
## Explore on your own

```python
# 1. Visualize any neuron you're curious about
可视化神经元('mixed4e', 100)
可视化神经元('mixed5b', 0)

# 2. Scan a layer to see what features it has
for ch in range(0, 480, 48):
    可视化神经元('mixed3b', ch, 标题=f'mixed3b:{ch}')

# 3. Test orientation tuning for a neuron you found
角度数组, 激活值数组 = 测量方向调谐(model, 'mixed3b', 200)
plt.polar(np.deg2rad(角度数组), 激活值数组)
plt.show()
```

---
## Further Reading

- [Zoom In: An Introduction to Circuits](https://distill.pub/2020/circuits/zoom-in/) — the paper this tutorial reproduces
- [Curve Detectors](https://distill.pub/2020/circuits/curve-detectors/) — in-depth analysis of curve detectors
- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) — feature superposition theory
- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) — mathematical framework for Transformer circuits
- [TransformerLens](https://github.com/neelnanda-io/TransformerLens) — Python toolkit for analyzing GPT-style models